In [2]:
import pandas as pd

import pickle
import re

Load our lexicons:

In [3]:
# Get negation and uncertainty cues
negation_esp = pd.read_csv("../data/lexicons/negation/negation_esp.csv")
negation_cat = pd.read_csv("../data/lexicons/negation/negation_cat.csv")
uncertainty_esp = pd.read_csv("../data/lexicons/uncertainty/uncertainty_esp.csv")
uncertainty_cat = pd.read_csv("../data/lexicons/uncertainty/uncertainty_cat.csv")

# Combine both
negation_cues = pd.concat([negation_esp, negation_cat])
uncertainty_cues = pd.concat([uncertainty_esp, uncertainty_cat])

# Remove duplicated cues
negation_cues = negation_cues.drop_duplicates()
uncertainty_cues = uncertainty_cues.drop_duplicates()

In [4]:
df_annotations = pickle.load(open("df_predictions.pkl", "rb"))

In [5]:
df_annotations.head()

,doc_index,doc_id,result_id,start,end,label,text,line_number,clean_text,text_length
0,0,19026587,ent0,448,451,NEG,no,0,no,3
1,0,19026587,ent1,451,467,NSCO,habitos toxicos.,0,NaN,16
2,0,19026587,ent2,286,298,NSCO,cistoscopia,2,NaN,12
3,0,19026587,ent3,305,314,NEG,negativa,2,negativa,9
4,0,19026587,ent4,314,337,NSCO,para lesiones malignas,2,NaN,23


In [ ]:
def preprocess_text(text):
    """Preprocess the text for rule-based analysis"""

    # it always starts in "motiu d'ingres" so just take it from there

    # Usar expresiones regulares para capturar lo que hay entre "hola" y "adiós"
    result = re.search(r"motiu d'ingres(.*)destinacio a l'alta", text)

    if result:
        captured_text = result.group(1).strip()
        # print(result.group(1).strip())  # Resultado sin espacios extra
    else:
        raise Exception ("No se encontró texto entre 'hola' y 'hoy'")


    # Handle special characters and standardize text
    final_text = captured_text.lower()

    # Add other preprocessing steps as needed

    return final_text

def tokenize_text(text):
    """Split text into tokens (words, punctuation)"""
    # You can use spaCy, NLTK, or custom tokenization
    import re
    tokens = re.findall(r'\w+|[^\w\s]', text)
    return tokens

def find_cues(tokens, cue_lexicon):
    """Find all instances of cues in the text"""
    cues = []
    cue_terms = cue_lexicon['term']
    for i, token in enumerate(tokens):
        if token in cue_terms.values:
            cues.append({"index": i, "token": token})
    return cues

# def find_cues(tokens, cue_lexicon, max_ngram=3):
#     """Find cues using n-gram approach, avoiding duplicates and overlaps."""
#     cues = []
#     text = ' '.join(tokens)
#     sorted_cues = sorted(cue_lexicon, key=len, reverse=True)

#     matched_indices = set()  # Keep track of token indices that have been matched

#     for cue in sorted_cues:
#         cue_tokens = cue.split()
#         cue_length = len(cue_tokens)

#         if cue_length > max_ngram:
#             continue

#         for i in range(len(tokens) - cue_length + 1):
#             # Skip if the current token index has already been matched
#             if i in matched_indices:
#                 continue

#             # Check if the current cue matches the tokens at this position
#             if tokens[i:i+cue_length] == cue_tokens:
#                 # Add the cue to the list of found cues
#                 cues.append({
#                     "index": i,
#                     "token": cue,
#                     "length": cue_length
#                 })

#                 # Mark all token indices covered by this cue as matched
#                 for j in range(i, i+cue_length):
#                     matched_indices.add(j)

#                 # Break to avoid detecting the same cue multiple times at the same position
#                 break
#     return cues


def determine_scope(tokens, cue_index, window_size=5, direction="forward"):
    """Determine the scope of a cue based on linguistic rules"""
    # Start with basic window
    if direction == "forward":
        start = cue_index + 1
        end = min(cue_index + window_size + 1, len(tokens))
    else:
        start = max(0, cue_index - window_size)
        end = cue_index

    # Refine scope based on punctuation
    for i in range(start, end):
        if tokens[i] in ['.', ',', ';', ':', '!', '?']:
            end = i
            break

    return tokens[start:end]

def apply_negex(text, negation_cues, window_size=5):
    """Apply NegEx algorithm to find negation cues and their scopes"""
    processed_text = preprocess_text(text)
    tokens = tokenize_text(processed_text)
    results = []

    # Find all negation cues
    cues = find_cues(tokens, negation_cues)

    # For each cue, determine its scope
    for cue in cues:
        # Default to looking forward for scope
        scope_tokens = determine_scope(tokens, cue["index"], window_size)

        # Store the result
        results.append({
            "cue": cue["token"],
            "cue_position": cue["index"],
            "scope": " ".join(scope_tokens),
            "scope_start": cue["index"] + 1,
            "scope_end": cue["index"] + len(scope_tokens) + 1
        })

    return results

def apply_uncertainty(text, uncertainty_cues, window_size=5):
    """Apply NegEx algorithm to find negation cues and their scopes"""
    processed_text = preprocess_text(text)
    tokens = tokenize_text(processed_text)

    results = []

    # Find all negation cues
    cues = find_cues(tokens, uncertainty_cues)

    # For each cue, determine its scope
    for cue in cues:
        # Default to looking forward for scope
        scope_tokens = determine_scope(tokens, cue["index"], window_size)

        # Store the result
        results.append({
            "cue": cue["token"],
            "cue_position": cue["index"],
            "scope": " ".join(scope_tokens),
            "scope_start": cue["index"] + 1,
            "scope_end": cue["index"] + len(scope_tokens) + 1
        })

    return results

def apply_linguistic_rules(tokens, cue_index):
    """Apply linguistic rules to refine scope detection"""
    # Example rule: Scope extends until the next punctuation
    scope_end = cue_index + 1
    while scope_end < len(tokens) and tokens[scope_end] not in ['.', ',', ';', ':', '!', '?']:
        scope_end += 1

    return scope_end


def evaluate_system(predictions, gold_standard):
    """Evaluate the system against gold standard annotations"""
    # Calculate precision, recall, F1 for cue detection
    # cue_precision =
    # Calculate precision, recall, F1 for scope detection
    # scope_precision =
    # Return evaluation metrics
    return {
        # "cue_precision": cue_precision,
        # "cue_recall": cue_recall,
        # "cue_f1": cue_f1,
        # "scope_precision": scope_precision,
        # "scope_recall": scope_recall,
        # "scope_f1": scope_f1
    }

def process_dataset(data):
    """Process the entire dataset with your rule-based system"""
    results = []

    for doc in data:
        text = doc["data"]["text"]

        # Apply NegEx for negation
        negation_results = apply_negex(text, negation_cues)

        # Apply similar algorithm for uncertainty
        uncertainty_results = apply_uncertainty(text, uncertainty_cues)

        # Combine results
        doc_results = {
            "doc_id": doc["data"]["id"],
            "negation_results": negation_results,
            "uncertainty_results": uncertainty_results
        }

        results.append(doc_results)

    return results